
# Gen AI 04 · How good are the Gen AI data-quality checks?

The landing notebook (`src/raw/03_land_raw_files.py`) seeds known defects into the raw files, so every Gen AI
finding can be scored against ground truth:

| Check (Lakeflow pipeline) | Gen AI function | Ground truth |
|---|---|---|
| `peril_mismatch` | `ai_classify` reads the free-text description and infers the peril | coded `cause_of_loss` in the file differs from the source system (`claims`) |
| `pii_in_free_text` | `ai_mask` redacts names, phone numbers and emails | the description contains the seeded contact details |

A second, independent model family then acts as an **LLM judge** of the `ai_query` explanations (step 4).

Output: precision, recall and F1 per check in `dq_genai_eval` (appended per run), plus the misses and false alarms
to inspect. This is what a data-quality owner needs before trusting an LLM in a production pipeline.

In [ ]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "loss_ratio_coworker")
dbutils.widgets.text("judge_endpoint", "databricks-claude-sonnet-5-5")
dbutils.widgets.text("judge_sample", "40")
JUDGE = dbutils.widgets.get("judge_endpoint")
SAMPLE = int(dbutils.widgets.get("judge_sample"))
CATALOG = dbutils.widgets.get("catalog") or spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = dbutils.widgets.get("schema")
FQ = f"`{CATALOG}`.`{SCHEMA}`"

## Ground truth vs Gen AI flags, claim by claim

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW genai_scored AS
SELECT a.claim_id, a.report_date, a.cause_of_loss AS coded_cause, c.cause_of_loss AS source_cause, a.ai_peril,
       s.loss_description, a.loss_description_masked,
       a.cause_of_loss <> c.cause_of_loss                                   AS truth_peril_mismatch,
       NOT a.peril_consistent                                               AS flag_peril_mismatch,
       s.loss_description RLIKE '@example\\\\.com|\\\\+44 7700'             AS truth_pii,
       a.pii_detected                                                       AS flag_pii
FROM {FQ}.silver_claims_ai_checked a
JOIN {FQ}.claims c USING (claim_id)
JOIN {FQ}.silver_claims s USING (claim_id)
""")
display(spark.sql("SELECT * EXCEPT (loss_description) FROM genai_scored WHERE truth_peril_mismatch OR flag_peril_mismatch OR truth_pii OR flag_pii LIMIT 50"))

| claim_id | report_date | coded_cause | source_cause | ai_peril | loss_description_masked | truth_peril_mismatch | flag_peril_mismatch | truth_pii | flag_pii |
|---|---|---|---|---|---|---|---|---|---|
| CLM-3075616 | 2026-11-02 | Third-Party PD | Third-Party PD | third-party property damage | Our vehicle damaged a third-party bus shelter in Dublin. Contact [MASKED] on [MASKED] or [MASKED] | False | False | True | True |
| CLM-3075712 | 2026-11-03 | Third-Party BI | Third-Party BI | bodily injury | Pedestrian injured when our delivery vehicle mounted the kerb in Leeds. Contact [MASKED] on [MASKED] or [MASKED] | False | False | True | True |
| CLM-3075742 | 2026-11-03 | Heavy Weather | Heavy Weather | heavy weather at sea | Heavy seas caused containers to shift and fall overboard en route to Munich. Contact [MASKED] on [MASKED] or [MASKED] | False | False | True | True |
| CLM-3075745 | 2026-11-03 | Third-Party PD | Third-Party PD | third-party property damage | Our vehicle damaged a third-party bus shelter in Utrecht. Contact [MASKED] on [MASKED] or [MASKED] | False | False | True | True |
| CLM-3075748 | 2026-11-03 | Third-Party BI | Third-Party BI | bodily injury | Pedestrian injured when our delivery vehicle mounted the kerb in Rotterdam. Contact [MASKED] on [MASKED] or [MASKED] | False | False | True | True |
| CLM-3075754 | 2026-11-03 | Theft | Collision | vehicle collision | Delivery van rear-ended at traffic lights in Nantes, bumper and tailgate damaged | True | True | False | False |
| CLM-3075755 | 2026-11-03 | Third-Party BI | Third-Party BI | bodily injury | [MASKED] injured when our delivery vehicle mounted the kerb in Lille | False | False | False | True |
| CLM-3075838 | 2026-11-04 | Collision | Collision | vehicle collision | Company car collided with a parked vehicle while reversing in Munich. Contact [MASKED] on [MASKED] or [MASKED] | False | False | True | True |
| CLM-3075865 | 2026-11-04 | Collision | Collision | vehicle collision | Company car collided with a parked vehicle while reversing in Zurich. Contact [MASKED] on [MASKED] or [MASKED] | False | False | True | True |
| CLM-3075874 | 2026-11-04 | Storm/Wind | Storm/Wind | storm or wind | Fallen tree after windstorm damaged the office facade in Munich. Contact [MASKED] on [MASKED] or [MASKED] | False | False | True | True |
| CLM-3075880 | 2026-11-04 | Third-Party BI | Third-Party BI | bodily injury | Cyclist reports neck injury after collision with fleet car in Porto. Contact [MASKED] on [MASKED] or [MASKED] | False | False | True | True |
| CLM-3075899 | 2026-11-05 | Cargo Damage | Cargo Damage | water damage | Container of electronics arrived water-damaged at the port near Bristol. Contact [MASKED] on [MASKED] or [MASKED] | False | False | True | True |
| CLM-3075901 | 2026-11-05 | Collision | Collision | vehicle collision | Company car collided with a parked vehicle while reversing in Stuttgart. Contact [MASKED] on [MASKED] or [MASKED] | False | False | True | True |
| CLM-3075927 | 2026-11-05 | Third-Party BI | Third-Party BI | bodily injury | Pedestrian injured when our delivery vehicle mounted the kerb in Lyon. Contact [MASKED] on [MASKED] or [MASKED] | False | False | True | True |
| CLM-3075958 | 2026-11-05 | Theft | Collision | vehicle collision | Delivery van rear-ended at traffic lights in Liège, bumper and tailgate damaged | True | True | False | False |
| CLM-3075977 | 2026-11-05 | Collision | Collision | vehicle collision | Delivery van rear-ended at traffic lights in Lyon, bumper and tailgate damaged. Contact [MASKED] on [MASKED] or [MASKED] | False | False | True | True |
| CLM-3075980 | 2026-11-05 | Hail | Hail | hail | Hailstones damaged roof tiles and skylights at the insured warehouse in Manchester. Contact [MASKED] on [MASKED] or [MASKED] | False | False | True | True |
| CLM-3075981 | 2026-11-05 | Vessel Collision | Vessel Collision | vessel collision | Feeder vessel struck the quay while berthing near [MASKED] | False | False | False | True |
| CLM-3075982 | 2026-11-05 | Third-Party PD | Third-Party PD | third-party property damage | Insured truck scraped a third party's gate while turning in Lyon. Contact [MASKED] on [MASKED] or [MASKED] | False | False | True | True |
| CLM-3076011 | 2026-11-05 | Third-Party PD | Third-Party PD | third-party property damage | Insured truck scraped a third party's gate while turning in [MASKED] | False | False | False | True |
| CLM-3076029 | 2026-11-06 | Cargo Theft | Cargo Theft | cargo theft | Trailer with pharmaceutical cargo stolen from a rest area near Manchester. Contact [MASKED] on [MASKED] or [MASKED] | False | False | True | True |
| CLM-3076053 | 2026-11-06 | Water Damage | Fire | fire | Kitchen fire spread to the dining area of the insured restaurant in Nantes | True | True | False | False |
| CLM-3076073 | 2026-11-06 | Third-Party BI | Third-Party BI | bodily injury | Cyclist reports neck injury after collision with fleet car in Zurich. Contact [MASKED] on [MASKED] or [MASKED] | False | False | True | True |
| CLM-3076076 | 2026-11-06 | Theft | Theft | cargo theft | Thieves cut the fence at the Antwerp yard and removed copper cable | False | True | False | False |
| CLM-3076086 | 2026-11-06 | Theft | Theft | theft | Thieves cut the fence at the [MASKED] yard and removed copper cable | False | False | False | True |
| CLM-3076106 | 2026-11-06 | Bodily Injury | Bodily Injury | bodily injury | [MASKED] fell from a ladder on our Rotterdam site, serious back injury | False | False | False | True |
| CLM-3076108 | 2026-11-06 | Third-Party BI | Third-Party BI | bodily injury | [MASKED] injured when our delivery vehicle mounted the kerb in Stuttgart | False | False | False | True |
| CLM-3076113 | 2026-11-06 | Bodily Injury | Bodily Injury | bodily injury | [MASKED] slipped on a wet floor at the Liège premises and fractured a wrist | False | False | False | True |
| CLM-3076114 | 2026-11-06 | Water Damage | Water Damage | water damage | Leaking sprinkler head soaked packaged goods in the Munich warehouse. Contact [MASKED] on [MASKED] or [MASKED] | False | False | True | True |
| CLM-3076144 | 2026-11-07 | Collision | Collision | vehicle collision | Company car collided with a parked vehicle while reversing in Leeds. Contact [MASKED] on [MASKED] or [MASKED] | False | False | True | True |

_20 more rows in the CSV._

## Precision, recall, F1

In [ ]:
ev = spark.sql(f"""
WITH m AS (
  SELECT 'peril_mismatch' AS check_name, 'ai_classify' AS genai_function, truth_peril_mismatch AS t, flag_peril_mismatch AS f FROM genai_scored
  UNION ALL
  SELECT 'pii_in_free_text', 'ai_mask', truth_pii, flag_pii FROM genai_scored
)
SELECT current_timestamp() AS evaluated_at, check_name, genai_function, COUNT(*) AS claims_checked,
       SUM(CAST(t AS INT)) AS seeded_defects,
       SUM(CAST(f AS INT)) AS flagged,
       SUM(CAST(t AND f AS INT)) AS true_positives,
       SUM(CAST(f AND NOT t AS INT)) AS false_positives,
       SUM(CAST(t AND NOT f AS INT)) AS false_negatives,
       ROUND(SUM(CAST(t AND f AS INT)) / NULLIF(SUM(CAST(f AS INT)), 0), 3) AS precision,
       ROUND(SUM(CAST(t AND f AS INT)) / NULLIF(SUM(CAST(t AS INT)), 0), 3) AS recall
FROM m GROUP BY check_name, genai_function
""")
ev = ev.selectExpr("*", "ROUND(2 * precision * recall / NULLIF(precision + recall, 0), 3) AS f1")
ev.write.mode("append").option("mergeSchema", "true").saveAsTable(f"{FQ}.dq_genai_eval")
spark.sql(f"COMMENT ON TABLE {FQ}.dq_genai_eval IS 'Precision/recall of the Gen AI data-quality checks (ai_classify, ai_mask) against seeded defects, one row per check per run.'")
display(ev)

| evaluated_at | check_name | genai_function | claims_checked | seeded_defects | flagged | true_positives | false_positives | false_negatives | precision | recall | f1 |
|---|---|---|---|---|---|---|---|---|---|---|---|
| 2026-11-16T05:20:36.222Z | peril_mismatch | ai_classify | 1981 | 21 | 23 | 21 | 2 | 0 | 0.913 | 1.0 | 0.955 |
| 2026-11-16T05:20:36.222Z | pii_in_free_text | ai_mask | 1981 | 70 | 86 | 70 | 16 | 0 | 0.814 | 1.0 | 0.897 |

## Misses and false alarms (what to tune)
False alarms on `peril_mismatch` are usually narratives that fit two perils (e.g. a storm that caused water
damage); tune the label list or the peril → cause mapping in the pipeline rather than the threshold.

In [ ]:
display(spark.sql("""
SELECT 'peril false alarm' AS kind, claim_id, coded_cause, ai_peril, loss_description_masked FROM genai_scored WHERE flag_peril_mismatch AND NOT truth_peril_mismatch
UNION ALL
SELECT 'peril miss', claim_id, coded_cause, ai_peril, loss_description_masked FROM genai_scored WHERE truth_peril_mismatch AND NOT flag_peril_mismatch
UNION ALL
SELECT 'pii miss', claim_id, coded_cause, ai_peril, loss_description_masked FROM genai_scored WHERE truth_pii AND NOT flag_pii
LIMIT 30
"""))

| kind | claim_id | coded_cause | ai_peril | loss_description_masked |
|---|---|---|---|---|
| peril false alarm | CLM-3076076 | Theft | cargo theft | Thieves cut the fence at the Antwerp yard and removed copper cable |
| peril false alarm | CLM-3076151 | Theft | cargo theft | Thieves cut the fence at the Rotterdam yard and removed copper cable |

## LLM judge: are the ai_query explanations right, actionable and safe?
The explanations are written by the pipeline's model (Llama 3.3 70B). A different model family judges a sample of
them on four criteria, so the generator is not grading itself. Results are appended to `dq_explanation_judge`.

In [ ]:
judged = spark.sql(f"""
WITH s AS (
  SELECT claim_id, check_name, cause_of_loss, ai_peril, loss_description_masked, ai_explanation
  FROM {FQ}.dq_claims_issues ORDER BY hash(claim_id, check_name) LIMIT {SAMPLE}
),
j AS (
  SELECT *, ai_query('{JUDGE}', concat(
    'You are auditing an automated claims data-quality assistant. Judge the EXPLANATION below. ',
    'Reply with JSON only, no prose: {{"correct_check": true|false, "actionable": true|false, "no_personal_data": true|false, ',
    '"faithful": true|false, "score": 1-5, "reason": "<max 20 words>"}}. ',
    'correct_check: it explains the failed check named. actionable: it tells a claims handler what to change. ',
    'no_personal_data: it contains no names, phone numbers or emails. faithful: it invents no facts beyond the inputs. ',
    'CHECK: ', check_name, '. CODED CAUSE: ', coalesce(cause_of_loss, 'n/a'), '. AI PERIL: ', coalesce(ai_peril, 'n/a'),
    '. DESCRIPTION (masked): ', coalesce(loss_description_masked, ''), '. EXPLANATION: ', ai_explanation)) AS verdict_raw
  FROM s
)
SELECT current_timestamp() AS judged_at, '{JUDGE}' AS judge_endpoint, claim_id, check_name, ai_explanation,
       from_json(regexp_extract(verdict_raw, '[{{].*[}}]', 0),
                 'correct_check BOOLEAN, actionable BOOLEAN, no_personal_data BOOLEAN, faithful BOOLEAN, score INT, reason STRING') AS v
FROM j
""").selectExpr("judged_at", "judge_endpoint", "claim_id", "check_name", "ai_explanation", "v.*")
judged.write.mode("append").option("mergeSchema", "true").saveAsTable(f"{FQ}.dq_explanation_judge")
spark.sql(f"COMMENT ON TABLE {FQ}.dq_explanation_judge IS 'LLM-judge verdicts on ai_query data-quality explanations (independent model family), appended per run.'")
latest = f"(SELECT * FROM {FQ}.dq_explanation_judge WHERE judged_at = (SELECT MAX(judged_at) FROM {FQ}.dq_explanation_judge))"
display(spark.sql(f"""
SELECT check_name, COUNT(*) AS judged,
       ROUND(AVG(CAST(correct_check AS INT)), 3) AS correct_check, ROUND(AVG(CAST(actionable AS INT)), 3) AS actionable,
       ROUND(AVG(CAST(no_personal_data AS INT)), 3) AS no_personal_data, ROUND(AVG(CAST(faithful AS INT)), 3) AS faithful,
       ROUND(AVG(score), 2) AS avg_score
FROM {latest} GROUP BY ROLLUP (check_name) ORDER BY check_name NULLS LAST"""))
display(spark.sql(f"SELECT check_name, score, reason, ai_explanation FROM {latest} WHERE score <= 3 OR NOT faithful LIMIT 10"))

| check_name | judged | correct_check | actionable | no_personal_data | faithful | avg_score |
|---|---|---|---|---|---|---|
| negative_amount | 2 | 1.0 | 1.0 | 1.0 | 0.5 | 3.0 |
| non_eur_currency | 4 | 1.0 | 1.0 | 1.0 | 0.25 | 2.5 |
| peril_mismatch | 6 | 1.0 | 1.0 | 1.0 | 1.0 | 4.8 |
| pii_in_free_text | 28 | 0.857 | 1.0 | 1.0 | 0.893 | 3.46 |
|  | 40 | 0.897 | 1.0 | 1.0 | 0.821 | 3.51 |

| check_name | score | reason | ai_explanation |
|---|---|---|---|
| negative_amount | 3 | Correctly flags the negative incurred amount and says to make it positive, but ignores that negatives can be legitimate reserve reductions or recoveries. | The issue is that the incurred amount is negative (-23704.0), which is inconsistent with a legitimate claim. The claims handler should update the incurred amount to a positive value, as claims amounts cannot be negative. |
| pii_in_free_text | 2 | Explains a cause-code mismatch rather than the failed pii_in_free_text check; ignores masked contact details in the description. | The issue is that the coded cause of loss is "Cargo Damage" but the description suggests "water damage", indicating a potential inconsistency. The claims handler should update the coded cause of loss to "Water Damage" to |
| pii_in_free_text | 3 | Names the PII check and says to mask or remove it, but is generic. It ignores that the text is already masked and doesn't say what to change. | The issue is that the description field contains free text that may potentially include personally identifiable information (PII), which is not allowed. The claims handler should mask or remove any sensitive information  |
| pii_in_free_text | 3 | Addresses PII in free text and says to re-mask or remove it, but it is generic and ignores that the description is already masked. | The issue is that the description field contains free text that may potentially include personally identifiable information (PII), which is not allowed. The claims handler should re-mask or remove any sensitive informati |
| non_eur_currency | 2 | Names the non-EUR check, but Zurich is in Switzerland (CHF), so 'match location' is wrong; the amount and currency are not in the inputs. | The issue is that the incurred amount (41,225.0) is not in Euros, despite the location being Zurich, which suggests a European currency. The claims handler should update the currency of the incurred amount to Euros (EUR) |
| pii_in_free_text | 2 | Blames the city Stuttgart as PII, ignoring the masked personal identifier; advice is misdirected and the claim is speculative. | The issue is that the description field contains location-specific information (Stuttgart) which may be considered personal or sensitive data. The claims handler should remove or mask the location (Stuttgart) from the de |
| pii_in_free_text | 3 | Names the PII check and says to mask or remove it, but is generic and does not say what was flagged. | The issue is that the description field contains free text that may potentially include personally identifiable information (PII), which is not allowed. The claims handler should mask or remove any sensitive information  |
| negative_amount | 3 | Explains negative_amount and gives a fix, but the -23951.0 value and 'incurred' field are not in the inputs, and it overlooks legitimate negatives such as recoveries. | The issue is that the incurred amount is negative (-23951.0), which is inconsistent with a valid claim. The claims handler should update the incurred amount to a positive value, as claims amounts cannot be negative. |
| pii_in_free_text | 2 | Flags 'Vienna' as PII and ignores the masked contact details, which are the real PII; the fix is misdirected. | The issue is that the description contains location information ("Vienna") which may be considered personal identifiable information (PII). The claims handler should remove or mask the location to fix the issue and ensur |
| non_eur_currency | 2 | Addresses the currency check, but the amount 7450.0 is not in the inputs, and the fix is vague (convert or relabel?). | The issue is that the incurred amount of 7450.0 is not in Euros, as required. The claims handler should update the incurred amount to the correct currency, presumably Euros, to resolve the non_eur_currency check failure. |